In [ ]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("EventTimeWindows").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

# Event Time, Windows, and Watermarks

Streaming data carries two distinct notions of time:

| Time type | Definition | Example |
|-----------|-----------|--------|
| **Processing time** | When the event arrives at Spark | System clock at ingestion |
| **Event time** | When the event actually *happened* | Sensor measurement timestamp |

Using **event time** is essential for correct analytics because:
- Network delays, retries, or offline devices can deliver events late.
- A "page views per minute" metric should count by when views happened, not when they were ingested.

Structured Streaming handles event time with two mechanisms:
1. **`window(col, duration[, slide])`** -- groups events into fixed or sliding time buckets.
2. **`withWatermark(col, delay)`** -- tells Spark how long to wait for late data before closing a window.

**Java sources:** `AggregationsWithTimeWindow.java`, `StructuredStreamWordCount.java`, `DStreamAggregationRecap.java`, `IOTSensorSocket.java`

# Window Types

```
Tumbling window (non-overlapping):
  window("ts", "5 minutes")
  │◄─ 5 min ─►│◄─ 5 min ─►│◄─ 5 min ─►│
  [00:00─00:05][00:05─00:10][00:10─00:15]

Sliding window (overlapping):
  window("ts", "10 minutes", "5 minutes")   # 10-min window, slides every 5 min
  │◄──── 10 min ────►│
       │◄──── 10 min ────►│
            │◄──── 10 min ────►│
  [00:00─00:10]  event e1 appears in windows [00:00] and [23:50] if ts = 00:05
```

**Event time** is the timestamp embedded in the data (when the event *happened*),  
as opposed to **processing time** (when Spark *received* it).  
Using event time gives accurate results even when data arrives late or out of order.

**Tumbling window**: non-overlapping intervals  
`window("timestamp", "5 seconds")` -- one bucket per 5-second period

**Watermark**: tells Spark how late data can arrive  
`.withWatermark("timestamp", "10 seconds")` -- drop data older than 10 seconds  
This lets Spark clear old window state from memory.

With `withWatermark("ts", "10 seconds")`, Spark will:
- Track the **maximum event timestamp** seen so far (`max_ts`).
- Drop any events with `ts < max_ts - 10 seconds`.
- Emit (and garbage-collect) windows whose end time is before `max_ts - 10 seconds`.

# Example 1: IOT Sensor -- Event Time Tumbling Window

Sensor readings arrive as CSV: `sensorID, timestamp, temperature`.  
We want the **average temperature per sensor per 5-second window**.  
A 10-second watermark tolerates slightly delayed readings.

**Java source:** `AggregationsWithTimeWindow.java`, `IOTSensorSocket.java`

In [ ]:
# Simulate IOT sensor data (what IOTSensorSocket.java generates)
import datetime, random

random.seed(42)
BASE_TS = datetime.datetime(2024, 1, 1, 0, 0, 0)
IOT_DIR = "tmp/iot_stream"
os.makedirs(IOT_DIR, exist_ok=True)

def make_sensor_batch(start_offset_s, n_readings=20, n_sensors=3):
    lines = []
    for i in range(n_readings):
        for sensor_id in range(1, n_sensors + 1):
            ts = BASE_TS + datetime.timedelta(seconds=start_offset_s + i)
            temp = random.gauss(66, 5)
            lines.append(f"{sensor_id},{ts.strftime("%Y-%m-%d %H:%M:%S")},{temp:.2f}")
    return lines

batch = make_sensor_batch(0, n_readings=30)
with open(f"{IOT_DIR}/sensors_batch_01.csv", "w") as f:
    f.write("\n".join(batch) + "\n")

print(f"Generated {len(batch)} sensor readings.")
for l in batch[:5]:
    print(l)

In [ ]:
# Define schema for the sensor CSV
sensor_schema = T.StructType([
    T.StructField("sensorID",    T.IntegerType()),
    T.StructField("timestamp",   T.TimestampType()),
    T.StructField("temperature", T.DoubleType()),
])

# Batch read to explore shape
sensor_df = spark.read.schema(sensor_schema).csv(IOT_DIR)
sensor_df.show(5)
sensor_df.printSchema()

In [ ]:
# Tumbling window aggregation using Structured Streaming
# (trigger(availableNow=True) processes all existing files, then stops)

import shutil
CKPT = "tmp/iot_ckpt"
OUT  = "tmp/iot_out"
for d in (CKPT, OUT):
    if os.path.exists(d): shutil.rmtree(d)

iot_stream = spark.readStream.schema(sensor_schema).csv(IOT_DIR)

windowed_avg = (
    iot_stream
    .withWatermark("timestamp", "10 seconds")      # tolerate 10-s late arrivals
    .groupBy(
        F.col("sensorID"),
        F.window(F.col("timestamp"), "5 seconds")  # 5-second tumbling window
    )
    .agg(F.round(F.avg("temperature"), 2).alias("avg_temp"),
         F.count("*").alias("readings"))
    .select(
        F.col("sensorID"),
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("avg_temp"),
        F.col("readings"),
    )
)

q = (
    windowed_avg.writeStream
    .format("csv")
    .outputMode("append")
    .option("path", OUT)
    .option("header", "true")
    .option("checkpointLocation", CKPT)
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

result = spark.read.option("header","true").csv(OUT)
result.orderBy("sensorID", "window_start").show(20, truncate=False)

# Example 2: Sliding Window -- Word Count Over Time

A **sliding window** allows each event to count in multiple overlapping windows.  
Here we count words per 10-second sliding window, recalculated every 5 seconds.

**Java source:** `StructuredStreamWordCount.java`

> In the Java code, `current_timestamp()` is used as the event time (i.e. processing time).  
> Here we use it too since the poem data has no embedded timestamps.

In [ ]:
# Prepare poem text as a file stream source
POEM_STREAM = "tmp/poem_sliding_stream"
POEM_OUT    = "tmp/poem_sliding_out"
POEM_CKPT   = "tmp/poem_sliding_ckpt"
for d in (POEM_STREAM, POEM_OUT, POEM_CKPT):
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

with open("data/Safadi_poem.txt") as f:
    poem_lines = [l.strip() for l in f if l.strip()]

batch_size = 4
for i in range(0, len(poem_lines), batch_size):
    chunk = poem_lines[i:i+batch_size]
    with open(f"{POEM_STREAM}/batch_{i:03d}.txt", "w") as f:
        f.write("\n".join(chunk) + "\n")

print(f"Prepared {len(os.listdir(POEM_STREAM))} poem batches.")

In [ ]:
# Sliding window word count
poem_stream = spark.readStream.format("text").load(POEM_STREAM)

# Add a processing-time timestamp (since poem lines have none)
poem_ts = poem_stream.withColumn("ingest_ts", F.current_timestamp())

# Explode lines into words; count with a sliding window
word_window_count = (
    poem_ts
    .select(
        F.explode(F.split(F.col("value"), r"\s+")).alias("word"),
        F.col("ingest_ts"),
    )
    .filter(F.col("word") != "")
    .withWatermark("ingest_ts", "30 seconds")
    .groupBy(
        F.window(F.col("ingest_ts"), "30 seconds", "15 seconds"),
        F.col("word"),
    )
    .count()
    .orderBy(F.col("count").desc())
)

q2 = (
    word_window_count.writeStream
    .format("csv")
    .outputMode("append")
    .option("path", POEM_OUT)
    .option("header", "true")
    .option("checkpointLocation", POEM_CKPT)
    .trigger(availableNow=True)
    .start()
)
q2.awaitTermination()

spark.read.option("header","true").csv(POEM_OUT).show(15, truncate=False)

# Example 3: Late Data and Watermarks in Depth

When events arrive late (after their window should have been emitted), Spark must decide whether to reopen the closed window or discard the event.  

The **watermark** controls this threshold:

```
withWatermark("event_ts", "10 seconds")
```

Means: after seeing event time `T`, Spark will:
- Keep windows that end after `T - 10 seconds`.
- Discard any event whose `event_ts < T - 10 seconds`.

**Output modes and watermarks:**

| Output mode | Works with watermark? | Notes |
|-------------|----------------------|-------|
| `append`    | Required | Only emits rows after their window is finalized. |
| `update`    | Optional | Emits updated partial results each trigger. |
| `complete`  | Not compatible | Requires unbounded state. |

> In `append` mode, you get **exactly one output row per window** (no updates), but rows arrive later -- after the watermark advances past the window end.

In [ ]:
# Demonstrate the effect of watermark on late data
import datetime

LATE_DIR  = "tmp/late_data_stream"
LATE_OUT  = "tmp/late_data_out"
LATE_CKPT = "tmp/late_data_ckpt"
for d in (LATE_DIR, LATE_OUT, LATE_CKPT):
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

from pyspark.sql import Row

late_schema = T.StructType([
    T.StructField("event_ts",   T.TimestampType()),
    T.StructField("sensorID",   T.IntegerType()),
    T.StructField("value",      T.DoubleType()),
])

def write_events(filename, events):
    lines = [f"{e['event_ts']},{e['sensorID']},{e['value']:.1f}" for e in events]
    with open(f"{LATE_DIR}/{filename}", "w") as f:
        f.write("\n".join(lines) + "\n")

# On-time events: all within the same 5-second window
BASE = datetime.datetime(2024, 1, 1, 0, 0, 0)
write_events("batch_ontime.csv", [
    {"event_ts": BASE + datetime.timedelta(seconds=1), "sensorID": 1, "value": 70.0},
    {"event_ts": BASE + datetime.timedelta(seconds=2), "sensorID": 1, "value": 72.0},
    {"event_ts": BASE + datetime.timedelta(seconds=3), "sensorID": 1, "value": 68.0},
])

# Slightly late: arrives after the window but within the 10-s watermark
write_events("batch_late_ok.csv", [
    {"event_ts": BASE + datetime.timedelta(seconds=4), "sensorID": 1, "value": 75.0},
    {"event_ts": BASE + datetime.timedelta(seconds=20),"sensorID": 1, "value": 80.0},  # max_ts advances
])

# Very late: arrives after watermark -- will be DROPPED
write_events("batch_too_late.csv", [
    {"event_ts": BASE + datetime.timedelta(seconds=2), "sensorID": 1, "value": 99.9},  # too late!
    {"event_ts": BASE + datetime.timedelta(seconds=25),"sensorID": 1, "value": 82.0},
])

# Process with update mode (see partial results)
late_stream = spark.readStream.schema(late_schema).csv(LATE_DIR)
windowed = (
    late_stream
    .withWatermark("event_ts", "10 seconds")
    .groupBy(F.window(F.col("event_ts"), "5 seconds"), F.col("sensorID"))
    .agg(F.count("*").alias("count"), F.avg("value").alias("avg_val"))
    .select("sensorID", "window.start", "window.end", "count",
            F.round("avg_val", 1).alias("avg_val"))
)

q3 = (
    windowed.writeStream
    .format("csv")
    .outputMode("append")          # emit only after window is finalized
    .option("path", LATE_OUT)
    .option("header", "true")
    .option("checkpointLocation", LATE_CKPT)
    .trigger(availableNow=True)
    .start()
)
q3.awaitTermination()

result = spark.read.option("header","true").csv(LATE_OUT)
print("Final windowed output (late events beyond watermark were dropped):")
result.show(truncate=False)

# Shutdown Spark when done

In [ ]:
spark.stop()